# Fine-Tuning Whisper with Data Augmentation + LoRA (Parameter-Efficient Fine-Tuning)This is an upgraded version of your fine-tuning notebook. Two additions:1. Data augmentation - creates extra training examples from your existing recordings   by slightly varying speed and pitch, expanding your effective training data without   needing new recordings.2. LoRA (Low-Rank Adaptation) - instead of updating all 244 million parameters of   Whisper, this trains a small set of "adapter" weights (usually under 1% of the   total), which is much better suited to a small personal dataset like yours and   avoids the model forgetting its general speech knowledge.Run cells in order, top to bottom.

## Step 1 - GPU checkRuntime -> Change runtime type -> T4 GPU -> Save.

## Step 2 - Install required tools`peft` is Hugging Face's library for LoRA and other parameter-efficient methods.

In [ ]:
!pip install -q transformers datasets evaluate jiwer accelerate soundfile librosa peft torchao --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 52.1 MB/s eta 0:00:00


## Step 3 - Connect to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
base_path = "/content/drive/MyDrive/capstone-cleft-speech-data"
wav_folder = f"{base_path}/converted_wav"
csv_path = f"{base_path}/tracking/dataset.csv"
print("WAV files found:", len(os.listdir(wav_folder)))

Mounted at /content/drive
WAV files found: 360


## Step 4 - Load your dataset and split it

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv(csv_path)
df['wav_filename'] = df['filename'].apply(lambda x: os.path.splitext(x)[0] + ".wav")
df['wav_path'] = df['wav_filename'].apply(lambda x: os.path.join(wav_folder, x))
df = df[df['wav_path'].apply(os.path.exists)].reset_index(drop=True)
print("Usable recordings:", len(df))

train_df, val_df = train_test_split(df, test_size=0.15, random_state=42)
print(f"Training on: {len(train_df)} recordings (before augmentation)")
print(f"Validating on: {len(val_df)} recordings (held out - never augmented, never trained on)")

Usable recordings: 360
Training on: 306 recordings (before augmentation)
Validating on: 54 recordings (held out - never augmented, never trained on)


## Step 5 - Data augmentation (training set only)Creates 2 extra versions of each TRAINING recording:- A slightly faster version (0.95x speed is NOT used here - we use pitch/speed shifts  that are small enough to still sound like natural speech, not distorted)- A slightly pitch-shifted versionThe validation set is deliberately left untouched - augmenting it would make yourevaluation numbers artificially easier and not trustworthy.

In [ ]:
import librosa
import soundfile as sf
import numpy as np
import os
import pandas as pd # Ensure pandas is imported for DataFrame operations

augmented_folder = f"{base_path}/augmented_wav"
os.makedirs(augmented_folder, exist_ok=True)
augmented_rows = []
for idx, row in train_df.iterrows():
  y, sr = librosa.load(row['wav_path'], sr=16000)

  # Version 1: speed perturbation (slightly faster)
  y_speed = librosa.effects.time_stretch(y, rate=1.1)
  speed_name = row['wav_filename'].replace(".wav", "_aug_speed.wav")
  speed_path = os.path.join(augmented_folder, speed_name)
  sf.write(speed_path, y_speed, sr)
  augmented_rows.append({"wav_path": speed_path, "intended_text": row['intended_text']})

  # Version 2: slight pitch shift (does not change speed)
  y_pitch = librosa.effects.pitch_shift(y, sr=sr, n_steps=1.5)
  pitch_name = row['wav_filename'].replace(".wav", "_aug_pitch.wav")
  pitch_path = os.path.join(augmented_folder, pitch_name)
  sf.write(pitch_path, y_pitch, sr)
  augmented_rows.append({"wav_path": pitch_path, "intended_text": row['intended_text']})

augmented_df = pd.DataFrame(augmented_rows)

# Combine original training data with augmented versions
train_df_full = pd.concat([
    train_df[['wav_path', 'intended_text']],
    augmented_df], ignore_index=True)

print(f"Original training recordings: {len(train_df)}")
print(f"Augmented recordings added: {len(augmented_df)}")
print(f"Total training set size now: {len(train_df_full)}")

Original training recordings: 306
Augmented recordings added: 612
Total training set size now: 918


## Step 6 - Load the Whisper model and processor

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

model_name = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(model_name, language="English", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(model_name)

model.generation_config.language = "english"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  967MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

## Step 7 - Apply LoRA (parameter-efficient fine-tuning)This freezes the original 244 million parameters and adds small trainable adapterlayers on top - only these adapters get trained. `r=8` controls the adapter's size(a common, well-tested default). `target_modules` specifies which parts of Whisper'sattention mechanism get adapters - q_proj and v_proj are the standard choice forWhisper fine-tuning.

In [ ]:
from peft import LoraConfig, get_peft_model
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 884,736 || all params: 242,619,648 || trainable%: 0.3647


**Check the output above.** You should see something like "trainable params: ~1-2 million|| all params: ~244 million || trainable%: under 1%". This confirms LoRA is active -you're training a tiny fraction of the full model, which is exactly the point.

## Step 8 - Prepare the data for training

In [ ]:
!pip install -q -U datasets pyarrow accelerate

In [ ]:
from datasets import Dataset, Audio
def to_hf_dataset(dataframe):
    ds = Dataset.from_dict({
        "audio": list(dataframe['wav_path']),
        "text": list(dataframe['intended_text'])
    })
    ds = ds.cast_column("audio", Audio(sampling_rate=16000))
    return ds

train_dataset = to_hf_dataset(train_df_full)

def prepare_example(batch):
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch

train_dataset = train_dataset.map(prepare_example, remove_columns=train_dataset.column_names)
val_dataset_hf = to_hf_dataset(val_df[['wav_path','intended_text']])
val_dataset_hf = val_dataset_hf.map(prepare_example, remove_columns=val_dataset_hf.column_names)

print("Data preparation done.")
print("Training examples (with augmentation):", len(train_dataset))
print("Validation examples (untouched):", len(val_dataset_hf))

Map:   0%|          | 0/918 [00:00<?, ? examples/s]

Map:   0%|          | 0/54 [00:00<?, ? examples/s]

Data preparation done.
Training examples (with augmentation): 918
Validation examples (untouched): 54


## Step 9 - Data collator (padding) - same as before, no changes needed

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # pad input features to max length
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # pad labels to max length
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # replace padding with -100 to ignore it in loss calculation
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # if bos token is appended in previous tokenization step, remove it to not double append
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

## Step 10 - WER metric for tracking progress during training

In [ ]:
import evaluate
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

## Step 11 - Training settingsNote: `load_best_model_at_end` is intentionally left OUT this time (this caused the"identical to base model" bug in your full fine-tuning run) - we just keep the finalepoch's weights directly. With LoRA, training is also faster and lighter on memory,so epochs are increased to 15 to make the most of your (now larger, augmented)training set.

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

output_dir = f"{base_path}/finetuned_whisper_lora"

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    warmup_steps=30,
    num_train_epochs=15,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    predict_with_generate=True,
    generation_max_length=128,
    logging_steps=10,
    save_total_limit=2,
    report_to=[],
    remove_unused_columns=False,
    label_names=["labels"],
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset_hf,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

## Step 12 - TrainLoRA training is generally faster and lighter than full fine-tuning, but with moreepochs and more data (from augmentation), still expect roughly 30-60 minutes. Do notclose the tab or let the session idle.

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Wer
1,3.118411,1.364277,43.055556
2,0.316556,0.382708,23.611111
3,0.149746,0.339212,22.916667
4,0.074095,0.330312,22.222222
5,0.063995,0.333130,21.180556
6,0.041628,0.343408,21.527778
7,0.016797,0.361493,20.486111
8,0.005184,0.355996,20.833333
9,0.005306,0.373594,20.833333
10,0.004835,0.371671,20.833333


[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The cu

TrainOutput(global_step=1725, training_loss=0.6196548687844821, metrics={'train_runtime': 3534.4735, 'train_samples_per_second': 3.896, 'train_steps_per_second': 0.488, 'total_flos': 3.9913642156032e+18, 'train_loss': 0.6196548687844821, 'epoch': 15.0})

## Step 13 - Verify training actually changed the model before savingSame safety check as before - confirms training genuinely happened before you saveand move on.

In [ ]:
base_check = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
trained_weight = model.base_model.model.model.encoder.conv1.weight.detach().cpu()
fresh_weight = base_check.model.encoder.conv1.weight.detach().cpu()
identical = torch.equal(trained_weight, fresh_weight)
print("Model still identical to pretrained base (should be False):", identical)
if identical:
    print("WARNING: something is wrong - do not proceed to save.")
else:
    diff = (trained_weight - fresh_weight).abs().mean()
    print("Good - weights have changed. Average difference:", diff.item())
    print("Safe to proceed to saving.")

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Model still identical to pretrained base (should be False): True


## Step 14 - Save your LoRA fine-tuned modelThis saves both the adapter weights and the processor. The saved folder will be muchsmaller than a full fine-tuned model, since only the small adapter layers are unique -this is expected and correct for LoRA.

In [ ]:
import librosa
import torch

test_file = f"{base_path}/converted_wav/carrot.wav"  # any file you like

audio, sr = librosa.load(test_file, sr=16000)
inputs = processor.feature_extractor(audio, sampling_rate=16000, return_tensors="pt").input_features.to(model.device)

model.eval()
with torch.no_grad():
    with_adapter_ids = model.generate(inputs, max_new_tokens=50)
    with_adapter_text = processor.tokenizer.decode(with_adapter_ids[0], skip_special_tokens=True)

    with model.disable_adapter():
        without_adapter_ids = model.generate(inputs, max_new_tokens=50)
        without_adapter_text = processor.tokenizer.decode(without_adapter_ids[0], skip_special_tokens=True)

print("WITH LoRA adapter:", with_adapter_text)
print("WITHOUT LoRA adapter (base model behavior):", without_adapter_text)
print("Are they different?", with_adapter_text != without_adapter_text)

[transformers] Both `max_new_tokens` (=50) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


WITH LoRA adapter: carrot
WITHOUT LoRA adapter (base model behavior):  Yeah, I know it.
Are they different? True


In [ ]:
model.save_pretrained(output_dir)
processor.save_pretrained(output_dir)
print(f"LoRA fine-tuned model saved to: {output_dir}")

LoRA fine-tuned model saved to: /content/drive/MyDrive/capstone-cleft-speech-data/finetuned_whisper_lora


## Step 15 - How to load this model later (for your baseline WER notebook)LoRA models load slightly differently from a normal fine-tuned model - you load thebase model first, then attach the adapter on top. Use this exact pattern in yourBaseline WER Test notebook's Step 6, instead of a plain pipeline() call:```pythonfrom transformers import WhisperForConditionalGeneration, WhisperProcessor, pipelinefrom peft import PeftModelbase_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")model_with_lora = PeftModel.from_pretrained(base_model, f"{base_path}/finetuned_whisper_lora")model_with_lora = model_with_lora.merge_and_unload()  # combines adapter into the base model for normal useprocessor = WhisperProcessor.from_pretrained(f"{base_path}/finetuned_whisper_lora")asr_pipeline = pipeline(    "automatic-speech-recognition",    model=model_with_lora,    tokenizer=processor.tokenizer,    feature_extractor=processor.feature_extractor,    device=device,    generate_kwargs={        "language": "en",        "task": "transcribe",        "max_new_tokens": 100,        "no_repeat_ngram_size": 3,        "temperature": 0.0,        "do_sample": False,    })```Copy this pattern into your baseline notebook to get your official after-WER usingthis LoRA model.